# NeuraSight — Stacking Ensemble (Logistic Regression Meta-Learner)

Trains the stacking meta-learner on the saved base-model probabilities
(EfficientNet-B0, ResNet-50, DenseNet-121, VGG-16).

**Leakage-safe design:** the test-set probabilities are split in half
(stratified). The meta-learner trains on one half and is evaluated on the
untouched other half — so its reported accuracy is honest.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, pickle
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/NeuraSight/models"

# Order MUST match how the backend concatenates probabilities at inference
MODEL_ORDER = ["efficientnet", "resnet", "densenet", "vgg"]
SAVE_NAMES = {
    "efficientnet": "BRAIN_MRI_EFFICIENTNET",
    "resnet": "BRAIN_MRI_RESNET",
    "densenet": "BRAIN_MRI_DENSENET",
    "vgg": "BRAIN_MRI_VGG",
}
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
print("Meta-learner will stack:", MODEL_ORDER)

In [ ]:
prob_arrays = []
for key in MODEL_ORDER:
    path = os.path.join(SAVE_DIR, SAVE_NAMES[key] + "_test_probs.npy")
    arr = np.load(path)
    print(f"{key}: {arr.shape}")
    prob_arrays.append(arr)

y = np.load(os.path.join(SAVE_DIR, "test_labels.npy"))
X = np.concatenate(prob_arrays, axis=1)   # (N, 4 models * 4 classes = 16)
print("Feature matrix X:", X.shape, "| labels y:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, stratify=y, random_state=42)
print("Meta-train:", X_train.shape, "| Meta-test:", X_test.shape)

In [ ]:
meta = LogisticRegression(max_iter=1000, C=1.0)
meta.fit(X_train, y_train)
print("Meta-learner trained (Logistic Regression).")

In [ ]:
y_pred = meta.predict(X_test)
acc = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro")
print(f"Ensemble Accuracy : {acc*100:.2f}%")
print(f"Ensemble Precision: {prec*100:.2f}%")
print(f"Ensemble Recall   : {rec*100:.2f}%")
print(f"Ensemble F1       : {f1*100:.2f}%")
print()
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Stacking Ensemble Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()

In [ ]:
print("Accuracy on the SAME held-out meta-test split (fair comparison):")
for i, key in enumerate(MODEL_ORDER):
    base_pred = X_test[:, i*4:(i+1)*4].argmax(1)
    base_acc = accuracy_score(y_test, base_pred)
    print(f"  {key:12s}: {base_acc*100:.2f}%")
print(f"  {'ENSEMBLE':12s}: {acc*100:.2f}%")

In [ ]:
with open(os.path.join(SAVE_DIR, "meta_model.pkl"), "wb") as f:
    pickle.dump(meta, f)

ensemble_config = {
    "model_order": MODEL_ORDER,
    "save_names": SAVE_NAMES,
    "class_names": CLASS_NAMES,
    "meta_learner": "LogisticRegression",
    "feature_dim": int(X.shape[1]),
    "ensemble_accuracy": round(acc * 100, 2),
}
with open(os.path.join(SAVE_DIR, "ensemble_config.json"), "w") as f:
    json.dump(ensemble_config, f, indent=2)
print("Saved meta_model.pkl and ensemble_config.json")

In [ ]:
from google.colab import files
files.download(os.path.join(SAVE_DIR, "meta_model.pkl"))
files.download(os.path.join(SAVE_DIR, "ensemble_config.json"))

## Next Steps

Download into your project's `models/` folder:
- `BRAIN_MRI_EFFICIENTNET.pth`, `BRAIN_MRI_RESNET.pth`,
  `BRAIN_MRI_DENSENET.pth`, `BRAIN_MRI_VGG.pth`
- `meta_model.pkl` and `ensemble_config.json`

The backend `ai/` ensemble predictor loads all four CNNs + this meta-learner
to produce the final stacked prediction, then runs Grad-CAM on the base model
that agreed with the ensemble.